In [4]:
import pandas as pd
listings=pd.read_csv("../Data/clean_listings.csv")

In [6]:
listings.shape

(40889, 40)

In [7]:
listings["revenue_per_night"]=(
    listings["estimated_revenue_l365d"]/
    listings["estimated_occupancy_l365d"]
)

In [8]:
listings["occupancy_rate"]=(listings["estimated_occupancy_l365d"]/365)*100

In [9]:
def revenue_tier(x):
    if x<5000:
        return "Low"
    elif x<15000:
        return "Medium"
    elif x<50000:
        return "High"
    else:
        return "Premium"
listings["revenue_tier"] = listings["estimated_revenue_l365d"].apply(revenue_tier)

In [10]:
listings["revenue_tier"].value_counts()

revenue_tier
Low        14938
Medium     13069
High       10796
Premium     2086
Name: count, dtype: int64

In [11]:
def occupancy_tier(x):
    if x<30:
        return "Low"
    elif x<60:
        return "Medium"
    elif x<85:
        return "High"
    else:
        return "Excellent"
listings["occupancy_tier"] = listings["occupancy_rate"].apply(occupancy_tier)

In [12]:
listings["occupancy_tier"].value_counts()

occupancy_tier
Low       26545
Medium     7536
High       6808
Name: count, dtype: int64

In [13]:
def host_tier(x):
    if x<2:
        return "New Host"
    elif x<5:
        return "Experienced"
    elif x<10:
        return "Veteran"
    else:
        return "Expert"
listings["host_tier"] = listings["host_experience_years"].apply(host_tier)

In [14]:
listings["host_tier"].value_counts()

host_tier
Expert         15516
Veteran        12623
Experienced     8848
New Host        3902
Name: count, dtype: int64

In [15]:
price_90 = listings["price"].quantile(0.90)
listings["luxury_property"] = (listings["price"] >= price_90)

In [16]:
listings["luxury_property"].value_counts()

luxury_property
False    36796
True      4093
Name: count, dtype: int64

In [17]:
listings["high_rated"] = (listings["review_scores_rating"] >= 4.8)

In [18]:
listings["high_demand"] = (listings["occupancy_rate"] >= 80)

In [19]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
listings[
[
    "rev_norm",
    "occ_norm",
    "rating_norm"
]
] = scaler.fit_transform(
    listings[
    [
        "estimated_revenue_l365d",
        "occupancy_rate",
        "review_scores_rating"
    ]
    ]
)

In [20]:
listings["investment_score"]=(
    0.5*listings["rev_norm"]+
    0.3*listings["occ_norm"]+
    0.2*listings["rating_norm"])*100

In [21]:
listings["investment_score"].quantile([0.25,0.5,0.75,0.9,0.95])

0.25    21.282312
0.50    26.727725
0.75    39.089841
0.90    51.279007
0.95    53.356649
Name: investment_score, dtype: float64

In [23]:
q1 = listings["investment_score"].quantile(0.25)
q2 = listings["investment_score"].quantile(0.50)
q3 = listings["investment_score"].quantile(0.75)

def investment_grade(x):
    if x >= q3:
        return "A"
    elif x >= q2:
        return "B"
    elif x >= q1:
        return "C"
    else:
        return "D"
listings["investment_grade"] = listings["investment_score"].apply(investment_grade)

In [26]:
listings["investment_score"].describe()

count    40889.000000
mean        30.819307
std         12.466144
min          0.017220
25%         21.282312
50%         26.727725
75%         39.089841
max         99.350000
Name: investment_score, dtype: float64

In [28]:
listings[listings["investment_grade"]=="A"].shape

(10223, 52)

In [27]:
listings["investment_grade"].value_counts()

investment_grade
A    10223
B    10222
D    10222
C    10222
Name: count, dtype: int64

In [24]:
neighbourhood_score = listings.groupby("neighbourhood_cleansed")["investment_score"].mean().sort_values(ascending=False)
neighbourhood_score.head(20)

neighbourhood_cleansed
Westminster               32.581176
Camden                    32.577082
Islington                 32.520720
Lambeth                   32.378670
Southwark                 32.160901
City of London            32.019965
Kensington and Chelsea    31.644791
Tower Hamlets             31.111857
Hammersmith and Fulham    31.026871
Hackney                   30.623372
Haringey                  30.493822
Brent                     30.490510
Richmond upon Thames      29.902790
Hounslow                  29.617278
Ealing                    29.515320
Waltham Forest            29.337907
Newham                    29.298673
Wandsworth                29.285872
Harrow                    29.175910
Greenwich                 29.119405
Name: investment_score, dtype: float64

In [29]:
listings.to_csv("../Data/airbnb_investment_dataset.csv",index=False)